In [8]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union
from shapely import wkt
import numpy as np

In [9]:
from shapely.geometry import Point
movement = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/movement_from_hotel.csv")
movement = movement.rename(columns = {"place_id_hotel":"place_id", "place_name_hotel":"name", 
                                      "latitude_visited":"latitude", "longitude_visited":"longitude"})

# Convert lat/lon to geometry (Point objects)
movement['geometry'] = movement.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)

# Create a GeoDataFrame
movement = gpd.GeoDataFrame(movement, geometry='geometry', crs="EPSG:4326")
movement

,place_id,name,main_category_visited,latitude,longitude,geometry
0,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Locatie voor live muziek,52.362162,4.883806,POINT (4.88381 52.36216)
1,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Haute-cuisinerestaurant,52.369279,4.884023,POINT (4.88402 52.36928)
2,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Haute-cuisinerestaurant,52.369279,4.884023,POINT (4.88402 52.36928)
3,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Broodjeswinkel,52.370469,4.883942,POINT (4.88394 52.37047)
4,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Snackbar,52.368827,4.884062,POINT (4.88406 52.36883)
...,...,...,...,...,...,...
881551,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Kunstmuseum,52.358076,4.881205,POINT (4.88121 52.35808)
881552,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Kunstmuseum,52.358011,4.879755,POINT (4.87976 52.35801)
881553,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Brasserie,52.355639,4.870725,POINT (4.87072 52.35564)
881554,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Bar,52.366458,4.900025,POINT (4.90002 52.36646)


### Movement

#### Movement from hotel

In [10]:
df = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/movement_from_hotel_full.csv")

df['movement_line'] = df.apply(
    lambda row: LineString([
        (row['longitude_hotel'], row['latitude_hotel']),
        (row['longitude_visited'], row['latitude_visited'])
    ]),
    axis=1
)

In [11]:
df = df[["place_id_hotel", "place_name_hotel", "Buurt_hotel", "latitude_hotel", "longitude_hotel", "place_id_visited", "place_name_visited", "Buurt_visited", "latitude_visited", 
         "longitude_visited", "movement_line"]]

In [12]:
# Group by hotel and visited place, and count visits for each combination
visit_counts = df.groupby(['place_id_hotel', 'place_id_visited']).size().reset_index(name='visit_count')

# Calculate total visits from each hotel
total_visits = df.groupby('place_id_hotel').size().reset_index(name='total_visits')

# Merge the two DataFrames
visit_counts = pd.merge(visit_counts, total_visits, on='place_id_hotel')

# Compute the percentage of visits from each hotel to each visited place
visit_counts['percentage'] = (visit_counts['visit_count'] / visit_counts['total_visits']) * 100

In [13]:
move = df.drop_duplicates("movement_line")

In [14]:
move = move.merge(visit_counts, on = ["place_id_hotel", "place_id_visited"], how = "left")

In [15]:
move_from_hotel = move

In [16]:
move_from_hotel

,place_id_hotel,place_name_hotel,Buurt_hotel,latitude_hotel,longitude_hotel,place_id_visited,place_name_visited,Buurt_visited,latitude_visited,longitude_visited,movement_line,visit_count,total_visits,percentage
0,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Felix Meritisbuurt,52.369359,4.884211,ChIJKYyxKekJxkcRLAhtiEY8skw,Paradiso,Leidsebuurt-Zuidoost,52.362162,4.883806,"LINESTRING (4.8842114 52.369359, 4.8838059 52....",6,2164,0.277264
1,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Felix Meritisbuurt,52.369359,4.884211,ChIJg8bf-sIJxkcR9mc7li6y48E,Restaurant Vinkeles,Felix Meritisbuurt,52.369279,4.884023,"LINESTRING (4.8842114 52.369359, 4.884023 52.3...",12,2164,0.554529
2,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Felix Meritisbuurt,52.369359,4.884211,ChIJd_yDtNAJxkcRko3UnBeygWA,Chun Cafe Berenstraat,Felix Meritisbuurt,52.370469,4.883942,"LINESTRING (4.8842114 52.369359, 4.8839422 52....",3,2164,0.138632
3,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Felix Meritisbuurt,52.369359,4.884211,ChIJrftMriIJxkcRmsYdm9XycFA,Fabel Friet Runstraat,Leidsegracht-Noord,52.368827,4.884062,"LINESTRING (4.8842114 52.369359, 4.8840617 52....",2,2164,0.092421
4,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Felix Meritisbuurt,52.369359,4.884211,ChIJC-3i8KUJxkcRcSLXwcxcC_Y,Sojubar Amsterdam | Korean Fried Chicken & Beer,Gerard Doubuurt,52.356467,4.890757,"LINESTRING (4.8842114 52.369359, 4.8907568 52....",1,2164,0.046211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286681,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Riekerpolder,52.341766,4.824797,ChIJq3cHNaUJxkcR6_CXMDdNUOw,coffeecompany,Oosterdokseiland,52.375878,4.907677,"LINESTRING (4.824796699999999 52.3417663, 4.90...",1,4524,0.022104
286682,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Riekerpolder,52.341766,4.824797,ChIJfa7F_bsJxkcRkTrq3fNR68Q,Restaurant Van Beeren,Nieuwmarkt,52.372073,4.902273,"LINESTRING (4.824796699999999 52.3417663, 4.90...",1,4524,0.022104
286683,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Riekerpolder,52.341766,4.824797,ChIJz3ekGvAJxkcRu55KgXmGGoo,Barpiazza,Johannes Vermeerbuurt,52.356008,4.880294,"LINESTRING (4.824796699999999 52.3417663, 4.88...",1,4524,0.022104
286684,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Riekerpolder,52.341766,4.824797,ChIJ4zptppUJxkcR4sv5hVcIlMs,Il Tramezzino,Haarlemmerbuurt-Oost,52.380314,4.891513,"LINESTRING (4.824796699999999 52.3417663, 4.89...",1,4524,0.022104


#### Movement to place

In [17]:
df = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/movement_from_hotel_full.csv")

In [18]:
# Group by visited place and hotel, and count visits for each combination
visit_counts_place = df.groupby(['place_id_visited', 'place_id_hotel']).size().reset_index(name='visit_count')

# Calculate total visits from each place
total_visits_place = df.groupby('place_id_visited').size().reset_index(name='total_visits')

# Merge the two DataFrames
visit_counts_place = pd.merge(visit_counts_place, total_visits_place, on='place_id_visited')

# Compute the percentage of visits from each hotel to each visited place
visit_counts_place['percentage'] = (visit_counts_place['visit_count'] / visit_counts_place['total_visits']) * 100

In [19]:
move_hotel = df.drop_duplicates(subset = ["place_id_visited", "place_id_hotel"])
move_hotel = move_hotel[["place_id_hotel", "place_name_hotel", "Buurt_hotel", "Wijk_hotel", "latitude_hotel", "longitude_hotel", "place_id_visited", "place_name_visited", 
                         "Buurt_visited", "Wijk_visited", "latitude_visited", "longitude_visited"]]

In [20]:
move_to_place = move_hotel.merge(visit_counts_place, on = ["place_id_visited", "place_id_hotel"])

#### Move from Cores

In [21]:
category = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/places_categorised.csv", index_col = 0)[["place_id", "category"]]
move_to_place_cat = move_to_place.merge(category, left_on = "place_id_visited", right_on = "place_id", how = "inner")
move_from_hotel_cat = move_from_hotel.merge(category, left_on = "place_id_visited", right_on = "place_id", how = "inner")


In [22]:
buurt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/INDELING_BUURT (1).csv", delimiter = ";")[["Buurt", "Wijk", "Stadsdeel", "WKT_LNG_LAT"]]

In [23]:
buurt_to_category_place = move_to_place_cat.groupby(["Buurt_hotel", "category"])[["visit_count"]].sum().reset_index()
buurt_to_category_place_sum_visit = buurt_to_category_place.groupby("Buurt_hotel")["visit_count"].sum().reset_index()
buurt_to_category_place = buurt_to_category_place.merge(buurt_to_category_place_sum_visit, on= "Buurt_hotel", how = "inner", suffixes=('', '_sum'))
buurt_to_category_place["percentage"] = buurt_to_category_place["visit_count"]/buurt_to_category_place["visit_count_sum"]*100
buurt_to_category_place = buurt.merge(buurt_to_category_place, left_on = "Buurt", right_on = "Buurt_hotel", how = "inner")

In [24]:
#buurt_to_category_place.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/buurt_to_category.csv")

In [47]:
buurt_to_category_place

,Buurt,Wijk,Stadsdeel,WKT_LNG_LAT,Buurt_hotel,category,visit_count,visit_count_sum,percentage
0,Leliegracht e.o.,Grachtengordel-West,Centrum,"POLYGON((4.883069 52.373992,4.883385 52.374065...",Leliegracht e.o.,Activities,30,2629,1.141118
1,Leliegracht e.o.,Grachtengordel-West,Centrum,"POLYGON((4.883069 52.373992,4.883385 52.374065...",Leliegracht e.o.,Cannabisshop,159,2629,6.047927
2,Leliegracht e.o.,Grachtengordel-West,Centrum,"POLYGON((4.883069 52.373992,4.883385 52.374065...",Leliegracht e.o.,Conference,13,2629,0.494485
3,Leliegracht e.o.,Grachtengordel-West,Centrum,"POLYGON((4.883069 52.373992,4.883385 52.374065...",Leliegracht e.o.,Cultural & Historic,731,2629,27.805249
4,Leliegracht e.o.,Grachtengordel-West,Centrum,"POLYGON((4.883069 52.373992,4.883385 52.374065...",Leliegracht e.o.,Dining & Cafes,1325,2629,50.399391
...,...,...,...,...,...,...,...,...,...
1452,Langestraat e.o.,Grachtengordel-West,Centrum,"POLYGON((4.886206 52.378089,4.889543 52.376955...",Langestraat e.o.,Entertainment & Nightlife,10,2274,0.439754
1453,Langestraat e.o.,Grachtengordel-West,Centrum,"POLYGON((4.886206 52.378089,4.889543 52.376955...",Langestraat e.o.,Event venue,53,2274,2.330695
1454,Langestraat e.o.,Grachtengordel-West,Centrum,"POLYGON((4.886206 52.378089,4.889543 52.376955...",Langestraat e.o.,Green Recreation,93,2274,4.089710
1455,Langestraat e.o.,Grachtengordel-West,Centrum,"POLYGON((4.886206 52.378089,4.889543 52.376955...",Langestraat e.o.,Hotel,12,2274,0.527704


In [25]:
dstrct = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/GEBIED_STADSDELEN.csv", delimiter = ";")[["Stadsdeelnaam", "WKT_LNG_LAT"]]

In [26]:
dstrct_category = buurt_to_category_place.groupby(["Stadsdeel", "category"])["visit_count"].sum().reset_index()
dstrct_category_sum = dstrct_category.groupby("Stadsdeel")["visit_count"].sum().reset_index()
dstrct_category = dstrct_category.merge(dstrct_category_sum, on= "Stadsdeel", how = "inner", suffixes=('', '_sum'))
dstrct_category["percentage"] = dstrct_category["visit_count"]/dstrct_category["visit_count_sum"]*100
dstrct_category = dstrct_category.merge(dstrct, left_on = "Stadsdeel", right_on = "Stadsdeelnaam", how = "inner")

In [32]:
dstrct_category.head(10)

,Stadsdeel,category,visit_count,visit_count_sum,percentage,Stadsdeelnaam,WKT_LNG_LAT
0,Centrum,Activities,4274,318600,1.341494,Centrum,"POLYGON((4.878167 52.378506,4.879862 52.38122,..."
1,Centrum,Cannabisshop,23436,318600,7.355932,Centrum,"POLYGON((4.878167 52.378506,4.879862 52.38122,..."
2,Centrum,Conference,2264,318600,0.710609,Centrum,"POLYGON((4.878167 52.378506,4.879862 52.38122,..."
3,Centrum,Cultural & Historic,91035,318600,28.573446,Centrum,"POLYGON((4.878167 52.378506,4.879862 52.38122,..."
4,Centrum,Dining & Cafes,143023,318600,44.891086,Centrum,"POLYGON((4.878167 52.378506,4.879862 52.38122,..."
5,Centrum,Entertainment & Nightlife,2026,318600,0.635907,Centrum,"POLYGON((4.878167 52.378506,4.879862 52.38122,..."
6,Centrum,Event venue,11592,318600,3.638418,Centrum,"POLYGON((4.878167 52.378506,4.879862 52.38122,..."
7,Centrum,Green Recreation,13810,318600,4.334589,Centrum,"POLYGON((4.878167 52.378506,4.879862 52.38122,..."
8,Centrum,Hotel,2011,318600,0.631199,Centrum,"POLYGON((4.878167 52.378506,4.879862 52.38122,..."
9,Centrum,Shopping,25129,318600,7.887320,Centrum,"POLYGON((4.878167 52.378506,4.879862 52.38122,..."


#### Cores

In [33]:
bijlmer_arena = ["Hoofdcentrum-Zuidoost", "Amstel III deel A/B-Noord", "Amsterdamse Poort"]
zuidas = ["Beatrixpark", "Zuidas-Noord", "RAI", "Zuidas-Zuid", "Vivaldi", "De Klenckebuurt"]
sloterdijk = ["Sloterdijk Stationskwartier"]

In [28]:
zuidas_core = buurt_to_category_place[buurt_to_category_place["Buurt_hotel"].isin(zuidas)]
zuidas_core = zuidas_core.groupby("category")[["visit_count"]].sum().reset_index()
zuidas_core["visit_count_sum"] = zuidas_core["visit_count"].sum()
zuidas_core["percentage"] = zuidas_core["visit_count"]/zuidas_core["visit_count_sum"]*100
zuidas_core.sort_values("percentage", ascending = False)

,category,visit_count,visit_count_sum,percentage
4,Dining & Cafes,8017,17269,46.424228
3,Cultural & Historic,4157,17269,24.072037
9,Shopping,1672,17269,9.682089
7,Green Recreation,945,17269,5.472233
6,Event venue,807,17269,4.673114
2,Conference,722,17269,4.180902
1,Cannabisshop,549,17269,3.179107
0,Activities,247,17269,1.430309
5,Entertainment & Nightlife,99,17269,0.573282
8,Hotel,54,17269,0.312699


In [27]:
sloterdijk_core = buurt_to_category_place[buurt_to_category_place["Buurt_hotel"].isin(sloterdijk)]
sloterdijk_core = sloterdijk_core.groupby("category")[["visit_count"]].sum().reset_index()
sloterdijk_core["visit_count_sum"] = sloterdijk_core["visit_count"].sum()
sloterdijk_core["percentage"] = sloterdijk_core["visit_count"]/sloterdijk_core["visit_count_sum"]*100
sloterdijk_core.sort_values("percentage", ascending = False)

,category,visit_count,visit_count_sum,percentage
4,Dining & Cafes,12850,32742,39.246228
3,Cultural & Historic,11127,32742,33.983874
9,Shopping,2553,32742,7.797325
1,Cannabisshop,2296,32742,7.012400
7,Green Recreation,1697,32742,5.182945
6,Event venue,1142,32742,3.487875
2,Conference,313,32742,0.955959
0,Activities,299,32742,0.913200
8,Hotel,283,32742,0.864333
5,Entertainment & Nightlife,182,32742,0.555861


In [26]:
bijlmer_core = buurt_to_category_place[buurt_to_category_place["Buurt_hotel"].isin(bijlmer_arena)]
bijlmer_core = bijlmer_core.groupby("category")[["visit_count"]].sum().reset_index()
bijlmer_core["visit_count_sum"] = bijlmer_core["visit_count"].sum()
bijlmer_core["percentage"] = bijlmer_core["visit_count"]/bijlmer_core["visit_count_sum"]*100
bijlmer_core.sort_values("percentage", ascending = False)

,category,visit_count,visit_count_sum,percentage
4,Dining & Cafes,13087,31652,41.346518
3,Cultural & Historic,8026,31652,25.357007
6,Event venue,3479,31652,10.991407
9,Shopping,3346,31652,10.571212
1,Cannabisshop,1452,31652,4.587388
7,Green Recreation,1293,31652,4.085050
0,Activities,375,31652,1.184759
2,Conference,319,31652,1.007835
5,Entertainment & Nightlife,153,31652,0.483382
8,Hotel,122,31652,0.385442


#### Buurt niveau

In [24]:
buurt_to_buurt_place

NameError: name 'buurt_to_buurt_place' is not defined

### Movement TravelTime

In [34]:
tt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/hotels_traveltime.csv", index_col = 0)[["place_id", "name", "walking_5min", "walking_15min", "walking_25min"]]

In [35]:
tt = tt.merge(df[["place_id_hotel", "Buurt_hotel"]].drop_duplicates("place_id_hotel"), left_on = "place_id", right_on = "place_id_hotel", how = "right")

In [36]:
tt_zuidas = tt[tt["Buurt_hotel"].isin(zuidas)]
tt_bijlmer_arena = tt[tt["Buurt_hotel"].isin(bijlmer_arena)]
tt_sloterdijk = tt[tt["Buurt_hotel"].isin(sloterdijk)]


In [37]:
##Zuidas

tt_zuidas['walking_5min']  = tt_zuidas['walking_5min'].apply(wkt.loads)
tt_zuidas['walking_15min'] = tt_zuidas['walking_15min'].apply(wkt.loads)
tt_zuidas['walking_25min'] = tt_zuidas['walking_25min'].apply(wkt.loads)

# 1. Compute the union for each column
tt_zuidas_5min  = unary_union(tt_zuidas['walking_5min'].tolist())
tt_zuidas_15min = unary_union(tt_zuidas['walking_15min'].tolist())
tt_zuidas_25min = unary_union(tt_zuidas['walking_25min'].tolist())

# 2. Pack them back into a new GeoDataFrame
new_tt_zuidas = gpd.GeoDataFrame({
    'core': ['zuidas', 'zuidas', 'zuidas'],
    'time': ['5 min','15 min','25 min'],
    'geometry':    [tt_zuidas_5min, tt_zuidas_15min, tt_zuidas_25min]
}, crs="EPSG:4326")


##Bijlmer ArenA

tt_bijlmer_arena['walking_5min']  = tt_bijlmer_arena['walking_5min'].apply(wkt.loads)
tt_bijlmer_arena['walking_15min'] = tt_bijlmer_arena['walking_15min'].apply(wkt.loads)
tt_bijlmer_arena['walking_25min'] = tt_bijlmer_arena['walking_25min'].apply(wkt.loads)

# 1. Compute the union for each column
tt_bijlmer_arena_5min  = unary_union(tt_bijlmer_arena['walking_5min'].tolist())
tt_bijlmer_arena_15min = unary_union(tt_bijlmer_arena['walking_15min'].tolist())
tt_bijlmer_arena_25min = unary_union(tt_bijlmer_arena['walking_25min'].tolist())

# 2. Pack them back into a new GeoDataFrame
new_tt_bijlmer_arena = gpd.GeoDataFrame({
    'core': ['bijlmer arena', 'bijlmer arena', 'bijlmer arena'],
    'time': ['5 min','15 min','25 min'],
    'geometry':    [tt_bijlmer_arena_5min, tt_bijlmer_arena_15min, tt_bijlmer_arena_25min]
}, crs="EPSG:4326")


##Sloterdijk

tt_sloterdijk['walking_5min']  = tt_sloterdijk['walking_5min'].apply(wkt.loads)
tt_sloterdijk['walking_15min'] = tt_sloterdijk['walking_15min'].apply(wkt.loads)
tt_sloterdijk['walking_25min'] = tt_sloterdijk['walking_25min'].apply(wkt.loads)

# 1. Compute the union for each column
tt_sloterdijk_5min  = unary_union(tt_sloterdijk['walking_5min'].tolist())
tt_sloterdijk_15min = unary_union(tt_sloterdijk['walking_15min'].tolist())
tt_sloterdijk_25min = unary_union(tt_sloterdijk['walking_25min'].tolist())

# 2. Pack them back into a new GeoDataFrame
new_tt_sloterdijk = gpd.GeoDataFrame({
    'core': ['sluiterdijk', 'sluiterdijk', 'sluiterdijk'],
    'time': ['5 min','15 min','25 min'],
    'geometry':    [tt_sloterdijk_5min, tt_sloterdijk_15min, tt_sloterdijk_25min]
}, crs="EPSG:4326")

C:\Users\isamu\AppData\Local\Temp\ipykernel_15940\1201739780.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tt_zuidas['walking_5min']  = tt_zuidas['walking_5min'].apply(wkt.loads)
C:\Users\isamu\AppData\Local\Temp\ipykernel_15940\1201739780.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tt_zuidas['walking_15min'] = tt_zuidas['walking_15min'].apply(wkt.loads)
C:\Users\isamu\AppData\Local\Temp\ipykernel_15940\1201739780.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

In [38]:
### Movement Cores only

In [39]:
move_cores = gpd.GeoDataFrame(move_to_place_cat, geometry=gpd.points_from_xy(move_to_place_cat.longitude_visited, move_to_place_cat.latitude_visited), crs="EPSG:4326")
move_zuidas = move_cores[move_cores["Buurt_hotel"].isin(zuidas)]
move_bijlmer_arena = move_cores[move_cores["Buurt_hotel"].isin(bijlmer_arena)]
move_sloterdijk = move_cores[move_cores["Buurt_hotel"].isin(sloterdijk)]

In [40]:
move_zuidas

,place_id_hotel,place_name_hotel,Buurt_hotel,Wijk_hotel,latitude_hotel,longitude_hotel,place_id_visited,place_name_visited,Buurt_visited,Wijk_visited,latitude_visited,longitude_visited,visit_count,total_visits,percentage,place_id,category,geometry
615,ChIJ-RAyRD0KxkcRG4koK51opDE,Hotel Novotel Amsterdam City,De Klenckebuurt,Buitenveldert-Oost,52.333716,4.888517,ChIJQyaHlxEKxkcRPNLV5M-b5zQ,Gelderlandplein,Gelderlandpleinbuurt,Buitenveldert-West,52.331024,4.877507,88,1990,4.422111,ChIJQyaHlxEKxkcRPNLV5M-b5zQ,Shopping,POINT (4.87751 52.33102)
791,ChIJ_dZhWgUKxkcRpOz1cKJp-vs,Crowne Plaza Amsterdam - South,Zuidas-Zuid,Zuidas,52.335791,4.874558,ChIJkweMYtMJxkcR19hz7lJVRVY,Westerpark,Westergasfabriek,Spaarndammerbuurt/Zeeheldenbuurt,52.386127,4.877835,15,3867,0.387898,ChIJkweMYtMJxkcR19hz7lJVRVY,Green Recreation,POINT (4.87784 52.38613)
792,ChIJ_dZhWgUKxkcRpOz1cKJp-vs,Crowne Plaza Amsterdam - South,Zuidas-Zuid,Zuidas,52.335791,4.874558,ChIJU92joJHhxUcRXySZ-NgjdRc,Park de Oeverlanden,Nieuwe Meer,Sloten/Nieuw-Sloten,52.335373,4.823437,2,361,0.554017,ChIJU92joJHhxUcRXySZ-NgjdRc,Green Recreation,POINT (4.82344 52.33537)
793,ChIJ_dZhWgUKxkcRpOz1cKJp-vs,Crowne Plaza Amsterdam - South,Zuidas-Zuid,Zuidas,52.335791,4.874558,ChIJ6VEZtInjxUcR_IsEYjlyzUw,Rosarium Vondelpark,Vondelpark-West,Willemspark,52.357625,4.863571,1,203,0.492611,ChIJ6VEZtInjxUcR_IsEYjlyzUw,Green Recreation,POINT (4.86357 52.35763)
794,ChIJ_dZhWgUKxkcRpOz1cKJp-vs,Crowne Plaza Amsterdam - South,Zuidas-Zuid,Zuidas,52.335791,4.874558,ChIJkVOGw17ixUcRFrLlMiacJcE,Hotel2Stay,Sloterdijk Stationskwartier,Sloterdijk Nieuw-West,52.386770,4.839570,1,203,0.492611,ChIJkVOGw17ixUcRFrLlMiacJcE,Hotel,POINT (4.83957 52.38677)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167988,ChIJ4eWiagQKxkcRf5j280yfbfE,INNSiDE by Melia Amsterdam,Zuidas-Noord,Zuidas,52.339882,4.870991,ChIJJ-S23XoJxkcRDsSNU2mgH34,Restaurant De Kas,Frankendael,Frankendael,52.352156,4.930511,1,578,0.173010,ChIJJ-S23XoJxkcRDsSNU2mgH34,Dining & Cafes,POINT (4.93051 52.35216)
167989,ChIJ4eWiagQKxkcRf5j280yfbfE,INNSiDE by Melia Amsterdam,Zuidas-Noord,Zuidas,52.339882,4.870991,ChIJdSivSz4JxkcRRdIHEML5QQ4,Leidseplein,Leidsebuurt-Zuidwest,De Weteringschans,52.364384,4.882747,1,495,0.202020,ChIJdSivSz4JxkcRRdIHEML5QQ4,Cultural & Historic,POINT (4.88275 52.36438)
167990,ChIJ4eWiagQKxkcRf5j280yfbfE,INNSiDE by Melia Amsterdam,Zuidas-Noord,Zuidas,52.339882,4.870991,ChIJxZjOFoIJxkcRR58iVcwqw-8,Sushi Fanatics,Oosterparkbuurt-Noordwest,Oosterparkbuurt,52.359301,4.913810,1,53,1.886792,ChIJxZjOFoIJxkcRR58iVcwqw-8,Dining & Cafes,POINT (4.91381 52.3593)
167991,ChIJ4eWiagQKxkcRf5j280yfbfE,INNSiDE by Melia Amsterdam,Zuidas-Noord,Zuidas,52.339882,4.870991,ChIJmf_gHk4JxkcRsUrmyHGinck,Massimo Gelato Oost,Transvaalbuurt-Oost,Transvaalbuurt,52.354364,4.920970,1,50,2.000000,ChIJmf_gHk4JxkcRsUrmyHGinck,Dining & Cafes,POINT (4.92097 52.35436)


In [41]:
# 1. for each time‐window, grab its polygon…
for window in ['5 min','15 min','25 min']:
    poly = new_tt_zuidas.loc[new_tt_zuidas['time']==window, 'geometry'].iloc[0]

    # 2. use .within() to test each point against that polygon
    #    and write out a column named inside_5min, etc.
    colname = f"inside_{window.replace(' ', '')}"
    move_zuidas[colname] = move_zuidas.geometry.within(poly)
    
# 1. for each time‐window, grab its polygon…
for window in ['5 min','15 min','25 min']:
    poly = new_tt_bijlmer_arena.loc[new_tt_bijlmer_arena['time']==window, 'geometry'].iloc[0]

    # 2. use .within() to test each point against that polygon
    #    and write out a column named inside_5min, etc.
    colname = f"inside_{window.replace(' ', '')}"
    move_bijlmer_arena[colname] = move_bijlmer_arena.geometry.within(poly)
    
# 1. for each time‐window, grab its polygon…
for window in ['5 min','15 min','25 min']:
    poly = new_tt_sloterdijk.loc[new_tt_sloterdijk['time']==window, 'geometry'].iloc[0]

    # 2. use .within() to test each point against that polygon
    #    and write out a column named inside_5min, etc.
    colname = f"inside_{window.replace(' ', '')}"
    move_sloterdijk[colname] = move_sloterdijk.geometry.within(poly)

C:\Users\isamu\AppData\Local\Programs\Python\Python313\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
C:\Users\isamu\AppData\Local\Programs\Python\Python313\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
C:\Users\isamu\AppData\Local\Programs\Python\Python313\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is tryin

In [42]:
zuidas_core_time = move_zuidas.groupby(["category"])[["visit_count", "inside_5min", "inside_15min", "inside_25min"]].sum().reset_index()
zuidas_core_time["percentage_inside_5min"] = zuidas_core_time["inside_5min"]/move_zuidas["inside_5min"].count()*100
zuidas_core_time["percentage_inside_15min"] = zuidas_core_time["inside_15min"]/move_zuidas["inside_15min"].count()*100
zuidas_core_time["percentage_inside_25min"] = zuidas_core_time["inside_25min"]/move_zuidas["inside_25min"].count()*100

In [43]:
bijlmer_arena_core_time = move_bijlmer_arena.groupby(["category"])[["visit_count", "inside_5min", "inside_15min", "inside_25min"]].sum().reset_index()
bijlmer_arena_core_time["percentage_inside_5min"] = bijlmer_arena_core_time["inside_5min"]/move_bijlmer_arena["inside_5min"].count()*100
bijlmer_arena_core_time["percentage_inside_15min"] = bijlmer_arena_core_time["inside_15min"]/move_bijlmer_arena["inside_15min"].count()*100
bijlmer_arena_core_time["percentage_inside_25min"] = bijlmer_arena_core_time["inside_25min"]/move_bijlmer_arena["inside_25min"].count()*100

In [44]:
sloterdijk_core_time = move_sloterdijk.groupby(["category"])[["visit_count", "inside_5min", "inside_15min", "inside_25min"]].sum().reset_index()
sloterdijk_core_time["percentage_inside_5min"] = sloterdijk_core_time["inside_5min"]/move_sloterdijk["inside_5min"].count()*100
sloterdijk_core_time["percentage_inside_15min"] = sloterdijk_core_time["inside_15min"]/move_sloterdijk["inside_15min"].count()*100
sloterdijk_core_time["percentage_inside_25min"] = sloterdijk_core_time["inside_25min"]/move_sloterdijk["inside_25min"].count()*100

In [45]:
def weighted_outside_pct(df, outside_col):
    # sum of weights for outside movements
    w_out = (df.loc[df[outside_col], 'visit_count']).sum()
    # total weight in this category
    w_tot = df['visit_count'].sum()
    return (w_out / w_tot) * 100 if w_tot else 0

## Zuidas
zuidas_pct = (
    move_zuidas
    .groupby('category')
    .apply(lambda grp: {
        'pct_inside_5min':  weighted_outside_pct(grp, 'inside_5min'),
        'pct_inside_15min': weighted_outside_pct(grp, 'inside_15min'),
        'pct_inside_25min': weighted_outside_pct(grp, 'inside_25min')
    })
    .apply(pd.Series)
    .reset_index()
)

## Bijlmer Arena
bijlmer_arena_pct = (
    move_bijlmer_arena
    .groupby('category')
    .apply(lambda grp: {
        'pct_inside_5min':  weighted_outside_pct(grp, 'inside_5min'),
        'pct_inside_15min': weighted_outside_pct(grp, 'inside_15min'),
        'pct_inside_25min': weighted_outside_pct(grp, 'inside_25min')
    })
    .apply(pd.Series)
    .reset_index()
)

## Sloterdijk
sloterdijk_pct = (
    move_sloterdijk
    .groupby('category')
    .apply(lambda grp: {
        'pct_inside_5min':  weighted_outside_pct(grp, 'inside_5min'),
        'pct_inside_15min': weighted_outside_pct(grp, 'inside_15min'),
        'pct_inside_25min': weighted_outside_pct(grp, 'inside_25min')
    })
    .apply(pd.Series)
    .reset_index()
)

C:\Users\isamu\AppData\Local\Temp\ipykernel_15940\2208351452.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grp: {
C:\Users\isamu\AppData\Local\Temp\ipykernel_15940\2208351452.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grp: {
C:\Users\isamu\AppData\Local\Temp\ipykernel_15940\2208351452.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping colu

## Final

In [46]:
sloterdijk_pct

,category,pct_inside_5min,pct_inside_15min,pct_inside_25min
0,Activities,0.000000,0.334448,1.337793
1,Cannabisshop,0.000000,8.362369,8.797909
2,Conference,25.239617,25.239617,25.239617
3,Cultural & Historic,0.000000,0.008987,0.512267
4,Dining & Cafes,11.595331,13.673152,15.167315
5,Entertainment & Nightlife,0.549451,0.549451,3.846154
6,Event venue,0.000000,2.539405,7.880911
7,Green Recreation,0.000000,0.000000,0.000000
8,Hotel,21.908127,28.975265,28.975265
9,Shopping,0.000000,0.117509,4.347826


In [143]:
bijlmer_core_full = bijlmer_core.merge(bijlmer_arena_pct, on = "category", how = "inner")
sloterdijk_core_full = sloterdijk_core.merge(sloterdijk_pct, on = "category", how = "inner")
zuidas_core_full = zuidas_core.merge(zuidas_pct, on = "category", how = "inner")

In [145]:
zuidas_core_full["core"] = "zuidas"
bijlmer_core_full["core"] = "bijlmer arena"
sloterdijk_core_full["core"] = "sloterdijk"

In [147]:
cores_full = pd.concat([zuidas_core_full, bijlmer_core_full, sloterdijk_core_full])

In [150]:
cores_full.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/cores_categories_visits.csv")

In [7]:
pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/cores_categories_visits.csv")

,Unnamed: 0,category,visit_count,visit_count_sum,percentage,pct_inside_5min,pct_inside_15min,pct_inside_25min,core
0,0,Activities,247,17269,1.430309,0.000000,4.453441,5.668016,zuidas
1,1,Cannabisshop,549,17269,3.179107,0.000000,5.646630,8.378871,zuidas
2,2,Conference,722,17269,4.180902,0.000000,99.722992,99.722992,zuidas
3,3,Cultural & Historic,4157,17269,24.072037,1.178735,1.178735,1.323070,zuidas
4,4,Dining & Cafes,8017,17269,46.424228,9.953848,14.219783,19.371336,zuidas
5,5,Entertainment & Nightlife,99,17269,0.573282,0.000000,0.000000,2.020202,zuidas
6,6,Event venue,807,17269,4.673114,0.000000,5.576208,11.400248,zuidas
7,7,Green Recreation,945,17269,5.472233,0.000000,29.206349,30.793651,zuidas
8,8,Hotel,54,17269,0.312699,0.000000,0.000000,5.555556,zuidas
9,9,Shopping,1672,17269,9.682089,0.358852,20.215311,22.308612,zuidas
